# Notebook 01: Pelatihan & Tuning Model Deteksi Wajah YOLOv8 (Epoch 0)

Notebook ini khusus menjalankan **Fokus 1**, yaitu membangun, melatih, dan men-tuning model detektor wajah **YOLOv8 Polosan (`arsitektur_yolov8_wajah.yaml`)** dari **Epoch 0** menggunakan **GPU NVIDIA GeForce RTX 4060 Laptop GPU** hingga menghasilkan bobot optimal:
- `storage/app/ai/weights/hasil_training_yolov8_wajah.pt`

Setelah file bobot `hasil_training_yolov8_wajah.pt` terbentuk dan mencapai skor `mAP` yang baik, proses dilanjutkan ke **Notebook 02 (`02_training_dataset_wajah_aktor/Notebook_02_Training_Dataset_Wajah_Aktor.ipynb`)**.

In [3]:
# Mengatur path absolut root proyek agar seluruh modul terbaca dengan benar dari Jupyter Notebook lokal
import sys
from pathlib import Path

FOLDER_NOTEBOOK_INI = Path.cwd().resolve()
if FOLDER_NOTEBOOK_INI.name == "01_training_model_yolo":
    FOLDER_ROOT_PROYEK = FOLDER_NOTEBOOK_INI.parent.parent
elif FOLDER_NOTEBOOK_INI.name == "training_python":
    FOLDER_ROOT_PROYEK = FOLDER_NOTEBOOK_INI.parent
else:
    FOLDER_ROOT_PROYEK = FOLDER_NOTEBOOK_INI

for path_modul in [
    FOLDER_ROOT_PROYEK,
    FOLDER_ROOT_PROYEK / "training_python",
    FOLDER_ROOT_PROYEK / "training_python" / "01_training_model_yolo",
    FOLDER_ROOT_PROYEK / "python",
]:
    if str(path_modul) not in sys.path:
        sys.path.insert(0, str(path_modul))

import konfigurasi_training as konfig
from persiapan_dataset_yolo import siapkan_dan_split_dataset_yolo
from training_dan_tuning_model_yolo import (
    muat_arsitektur_yolov8_polos,
    latih_model_yolov8_dari_nol,
    jalankan_tuning_hyperparameter_yolo,
    evaluasi_model_yolo_pada_split_test,
)

# Mengaktifkan mode output notebook unlimited dan memverifikasi penguncian GPU NVIDIA GeForce RTX 4060 Laptop GPU (GPU 1 pada Task Manager / cuda:0 pada PyTorch)
info_output_notebook = konfig.aktifkan_output_notebook_unlimited()
indeks_cuda, objek_device_gpu, info_perangkat_gpu = konfig.pilih_perangkat_gpu_rtx_wajib(wajib_cuda=True)
model_yolo_uji = muat_arsitektur_yolov8_polos()

print("Root Proyek          :", FOLDER_ROOT_PROYEK)
print("Perangkat GPU Aktif  :", info_perangkat_gpu["nama_gpu"], f"({info_perangkat_gpu['vram_gb']} GB VRAM)")
print("Indeks CUDA          :", info_perangkat_gpu["device_pytorch"], "-", info_perangkat_gpu["catatan_task_manager"])
print("Arsitektur YOLOv8    :", konfig.FILE_ARSITEKTUR_YOLOV8_YAML)
print("Jumlah Parameter YOLO:", sum(p.numel() for p in model_yolo_uji.model.parameters()))
print("Output Unlimited     :", info_output_notebook)

Root Proyek          : C:\laragon\www\IndoLens
Perangkat GPU Aktif  : NVIDIA GeForce RTX 4060 Laptop GPU (8.0 GB VRAM)
Indeks CUDA          : cuda:0 - Tampil sebagai GPU 1 di Windows Task Manager (GPU 0 adalah Intel UHD Graphics)
Arsitektur YOLOv8    : C:\laragon\www\IndoLens\python\config\arsitektur_yolov8_wajah.yaml
Jumlah Parameter YOLO: 11135987
Output Unlimited     : {'berjalan_di_notebook': True, 'jumlah_stream_diatur': 2, 'interval_flush_detik': 0.5, 'interval_progress_detik': 1.5, 'ultralytics_tqdm_diatur': True, 'tqdm_standar_diatur': True, 'konfig_server_jupyter_unlimited': True}


## Tahap 01: Persiapan, Validasi Label, & Pembagian Dataset Deteksi Wajah (`train / val / test`)
Mengambil gambar wajah manusia beserta label `.txt` dari `python/datasets/dataset_yolo_wajah/raw_wajah_manusia/` serta gambar negatif tanpa wajah (hewan/kera, benda, pemandangan) dari `raw_negatif_bukan_wajah/`, lalu membagi otomatis ke `split_yolo/` (`80% train`, `10% val`, `10% test`).

In [5]:
# Mengaktifkan mode output unlimited lalu menyiapkan dan membagi dataset deteksi wajah YOLOv8
konfig.aktifkan_output_notebook_unlimited()
ringkasan_persiapan_dataset_yolo = siapkan_dan_split_dataset_yolo(
    rasio_train=konfig.RASIO_SPLIT_TRAIN_YOLO,
    rasio_val=konfig.RASIO_SPLIT_VAL_YOLO,
    rasio_test=konfig.RASIO_SPLIT_TEST_YOLO,
)

  TAHAP 01: PERSIAPAN & PEMBAGIAN DATASET DETEKSI WAJAH YOLOV8
[*] Folder sumber wajah positif : C:\laragon\www\IndoLens\python\datasets\dataset_yolo_wajah\raw_wajah_manusia
[*] Folder sumber negatif/objek : C:\laragon\www\IndoLens\python\datasets\dataset_yolo_wajah\raw_negatif_bukan_wajah
[OK] Total sampel wajah positif       : 3000 gambar
[OK] Total sampel negatif (background): 550 gambar
     - Split TRAIN: 2840 gambar (Positif: 2400, Negatif: 440, Kotak Wajah: 8527)
     - Split VAL  : 355 gambar (Positif: 300, Negatif: 55, Kotak Wajah: 1025)
     - Split TEST : 355 gambar (Positif: 300, Negatif: 55, Kotak Wajah: 1121)
[OK] File konfigurasi YOLO ditulis ke : C:\laragon\www\IndoLens\python\datasets\dataset_yolo_wajah\split_yolo\data_yolo_wajah.yaml


## Tahap 02: Pelatihan & Tuning Model YOLOv8 Polosan dari Epoch 0 di GPU RTX 4060
Melatih `arsitektur_yolov8_wajah.yaml` dari bobot awal acak (`pretrained=False`, `amp=False`) dengan parameter tuning optimizer, *Cosine Learning Rate*, bobot *Loss* (`box`, `cls`, `dfl`), serta *Early Stopping* (`patience`), lalu menyimpan hasil terbaik ke `storage/app/ai/weights/hasil_training_yolov8_wajah.pt`.

In [7]:
# Mengaktifkan mode output unlimited lalu melatih model YOLOv8 deteksi wajah dari Epoch 0 menggunakan GPU RTX 4060
konfig.aktifkan_output_notebook_unlimited()
data_hasil_training_yolo = latih_model_yolov8_dari_nol(
    jumlah_epoch=konfig.JUMLAH_EPOCH_TRAINING_YOLO,
    batas_patience=konfig.BATAS_PATIENCE_EARLY_STOPPING_YOLO,
    ukuran_gambar=konfig.UKURAN_GAMBAR_INPUT_YOLO,
    ukuran_batch=konfig.UKURAN_BATCH_TRAINING_YOLO,
)

  TAHAP 02: PELATIHAN & TUNING YOLOV8 DETEKSI WAJAH DARI EPOCH 0 (NO PRE-TRAINED)
[*] Perangkat Komputasi GPU     : NVIDIA GeForce RTX 4060 Laptop GPU (8.0 GB VRAM)
[*] Indeks Perangkat            : CUDA:0 (Tampil sebagai GPU 1 di Windows Task Manager (GPU 0 adalah Intel UHD Graphics))
[*] File Arsitektur Polosan     : C:\laragon\www\IndoLens\python\config\arsitektur_yolov8_wajah.yaml
[*] File Dataset YOLO           : C:\laragon\www\IndoLens\python\datasets\dataset_yolo_wajah\split_yolo\data_yolo_wajah.yaml
[*] Target Epoch & Patience     : Maks 80 Epoch (Early Stopping Patience = 15)
[*] Ukuran Gambar & Batch       : 640x640 piksel | Batch Size = 8
[*] Optimizer & LR              : AdamW | lr0 = 0.001 | Cosine LR = True
[*] Bobot Loss (Box/Cls/DFL)    : box=7.5, cls=0.5, dfl=1.5
New https://pypi.org/project/ultralytics/8.4.215 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.113  Python-3.14.2 torch-2.14.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
e

### (Opsional) Eksperimen Perbandingan Hyperparameter Tuning YOLOv8
Menjalankan perbandingan beberapa skenario parameter (misal `AdamW` vs `SGD`) dari `DAFTAR_KANDIDAT_TUNING_YOLO` dan memilih bobot dengan skor `mAP50-95` tertinggi.

In [9]:
# Mengaktifkan mode output unlimited sebelum menjalankan eksperimen perbandingan kandidat tuning hyperparameter YOLOv8
konfig.aktifkan_output_notebook_unlimited()
# laporan_tuning_yolo = jalankan_tuning_hyperparameter_yolo(jumlah_epoch_per_skenario=30, batas_patience=10)

{'berjalan_di_notebook': True,
 'jumlah_stream_diatur': 2,
 'interval_flush_detik': 0.5,
 'interval_progress_detik': 1.5,
 'ultralytics_tqdm_diatur': True,
 'tqdm_standar_diatur': True,
 'konfig_server_jupyter_unlimited': True}

## Tahap 03: Evaluasi Model YOLOv8 pada Split Test
Menguji bobot `hasil_training_yolov8_wajah.pt` pada data uji (`split test`) untuk memperoleh skor `Precision`, `Recall`, `mAP50`, dan `mAP50-95`.

In [11]:
# Mengaktifkan mode output unlimited lalu mengevaluasi akurasi deteksi kotak wajah YOLOv8 pada split test
konfig.aktifkan_output_notebook_unlimited()
data_evaluasi_test_yolo = evaluasi_model_yolo_pada_split_test()
print(data_evaluasi_test_yolo)

Ultralytics 8.4.113  Python-3.14.2 torch-2.14.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
arsitektur_YOLOv8_wajah summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 89.284.0 MB/s, size: 621.3 KB)
val: Scanning C:\laragon\www\IndoLens\python\datasets\dataset_yolo_wajah\split_yolo\labels\test... 355 images, 55 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 355/355 608.2it/s 0.6s
val: New cache created: C:\laragon\www\IndoLens\python\datasets\dataset_yolo_wajah\split_yolo\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 45/45 13.1it/s 3.4s.9s
                   all        355       1121      0.854      0.627      0.699      0.433
Speed: 1.0ms preprocess, 4.6ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to C:\laragon\www\IndoLens\training_python\01_training_model_yolo\runs\detect\val-4
{'path_bobot_diuji': 'C